# 02. Hipoteza 1: Eskalacja emocjonalna

Sprawdzamy, czy link negatywny jest częstszy, gdy w poprzednich 24 godzinach ta sama para `SOURCE_SUBREDDIT -> TARGET_SUBREDDIT` miała wysokie natężenie `LIWC_Anger`.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks": #jesli uruchamiamy z katalogu notebooks, to przechodzimy o poziom wyżej
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path: #dodajemy projekt do sys.path, żeby można było importować moduły z projektu
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "database" / "NajnowszaWersjaBazy1205.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
#TESTUJEMY CZY ŚCIEŻKI SĄ POPRAWNE
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH exists:", DATA_PATH.exists())


PROJECT_ROOT: C:\Users\szymon\projekt_reddit
DATA_PATH exists: True


## Przygotowanie cech historycznych dla par subredditów

### Cel analizy

Celem analizy jest sprawdzenie hipotezy eskalacji emocjonalnej: czy prawdopodobieństwo wystąpienia negatywnego linku między dwoma subredditami wzrasta, jeżeli w ciągu poprzednich 24 godzin występowały między nimi interakcje o wysokim poziomie złości (`LIWC_Anger`).

### Co robi kod?

Dla każdego aktualnego linku kod sprawdza wcześniejsze interakcje pomiędzy tą samą parą subredditów w ciągu poprzednich 24 godzin.

W tej analizie interesuje nas relacja **między dwiema społecznościami**, dlatego kierunek linku nie ma znaczenia:

* interakcja `A → B`,
* interakcja `B → A`

są traktowane jako historia tej samej pary `A ↔ B`.

Na podstawie wcześniejszych interakcji kod tworzy trzy nowe cechy:

| Nowa kolumna                 | Znaczenie                                                                                |
| ---------------------------- | ---------------------------------------------------------------------------------------- |
| `prev_pair_interactions_24h` | Liczba wcześniejszych interakcji pomiędzy daną parą subredditów w ostatnich 24 godzinach |
| `prev_pair_mean_anger_24h`   | Średni poziom złości (`LIWC_Anger`) we wcześniejszych interakcjach tej pary              |
| `prev_pair_max_anger_24h`    | Najwyższy poziom złości (`LIWC_Anger`) we wcześniejszych interakcjach tej pary           |

### Przykład działania

Załóżmy, że pomiędzy subredditami `A` i `B` wystąpiły następujące interakcje:

| Czas interakcji | Kierunek linku | `LIWC_Anger` |
| --------------- | -------------- | -----------: |
| 10:00           | `A → B`        |        0.020 |
| 14:00           | `B → A`        |        0.030 |
| 18:00           | `A → B`        |        0.040 |

Dla linku z godziny **18:00** kod bierze pod uwagę obie wcześniejsze interakcje: zarówno `A → B` z godziny 10:00, jak i `B → A` z godziny 14:00. Oba linki opisują wcześniejszą historię kontaktu pomiędzy tymi samymi społecznościami.

Obliczone cechy dla linku z godziny 18:00 będą wyglądać następująco:

| Cecha                                 | Obliczenie             |   Wynik |
| ------------------------------------- | ---------------------- | ------: |
| Liczba wcześniejszych interakcji      | `2` wcześniejsze linki |     `2` |
| Średni wcześniejszy poziom złości     | `(0.020 + 0.030) / 2`  | `0.025` |
| Maksymalny wcześniejszy poziom złości | `max(0.020, 0.030)`    | `0.030` |

### Interpretacja

Dla linku z godziny 18:00 kod zapisze, że pomiędzy subredditami `A` i `B` wystąpiły w poprzednich 24 godzinach dwie wcześniejsze interakcje. Ich średni poziom złości wynosił `0.025`, a maksymalny poziom złości wynosił `0.030`.

Bieżący link z godziny 18:00 nie jest wliczany do własnej historii. Może natomiast zostać wykorzystany jako wcześniejsza interakcja przy analizie kolejnych linków pomiędzy tymi samymi subredditami.

### Zastosowanie

Tak przygotowane zmienne pozwalają sprawdzić, czy podwyższony poziom złości w niedawnych interakcjach między dwiema społecznościami wiąże się z większym prawdopodobieństwem wystąpienia kolejnego negatywnego linku.


In [ ]:
from scipy import stats #do testów statystycznych
from src.reddit_pbl.features import add_pair_history_features, sentiment_to_binary_negative
 #funkcja do dodawania cech związanych z historią pary komentarzy oraz funkcja do binarizacji sentymentu

 def sentiment_to_binary_negative(values: pd.Series) -> pd.Series:
    """Map LINK_SENTIMENT values to 1 for negative links and 0 otherwise."""
    return (values == -1).astype(int)

#historia pary komentarzy - liczba interakcji, średnia i maksymalna wartość LIWC_Anger w ciągu ostatnich 24h dla danej pary subredditów 
#(SOURCE_SUBREDDIT -> TARGET_SUBREDDIT)
def add_pair_history_features(
    df: pd.DataFrame,   #bierze DataFrame z kolumnami TIMESTAMP, SOURCE_SUBREDDIT, TARGET_SUBREDDIT, LIWC_Anger
    *, #oznacza, że wszystkie argumenty po tym muszą być przekazywane jako słowa kluczowe
    timestamp_col: str = "TIMESTAMP", #nazwa kolumny z timestampem
    source_col: str = "SOURCE_SUBREDDIT", #nazwa kolumny z źródłowym subredditem
    target_col: str = "TARGET_SUBREDDIT", #nazwa kolumny z docelowym subredditem
    anger_col: str = "LIWC_Anger", #nazwa kolumny z wartością LIWC_Anger
    window_hours: int = 24, #okno czasowe w godzinach
) -> pd.DataFrame: #zwraca DataFrame z dodanymi kolumnami: prev_pair_interactions_24h, prev_pair_mean_anger_24h, prev_pair_max_anger_24h
    """Add prior 24h interaction and anger features for each directed subreddit pair."""
    result = df.copy()

    # Konwersja kolumny z timestampem na typ datetime i sortowanie danych po timestampie
    result[timestamp_col] = pd.to_datetime(result[timestamp_col]) 
    result = result.sort_values(timestamp_col).reset_index(drop=True) #sortujemy od najstarszych do , ustawiamy nowy indeks
    
    # Słownik do przechowywania historii interakcji dla każdej pary subredditów
    histories: dict[tuple[str, str], deque[tuple[pd.Timestamp, float]]] = defaultdict(deque) #deque pozwala na szybkie dodawanie i usuwanie elementów z obu końców, idealny do przechowywania historii interakcji w oknie czasowym
    counts: list[int] = [] #lista do przechowywania liczby interakcji w oknie czasowym dla każdej pary subredditów
    mean_anger: list[float] = [] #lista do przechowywania średniej wartości LIWC_Anger w oknie czasowym dla każdej pary subredditów
    max_anger: list[float] = [] #lista do przechowywania maksymalnej wartości LIWC_Anger w oknie czasowym dla każdej pary subredditów
    window = pd.Timedelta(hours=window_hours)


#Iterujemy po wierszach DataFrame, dla każdego wiersza aktualizujemy historię interakcji dla danej pary subredditów i obliczamy cechy na podstawie historii 
    for row in result.itertuples(index=False):
        timestamp = getattr(row, timestamp_col) #GETATTR pozwala na dynamiczne pobieranie wartości z wiersza na podstawie nazwy kolumny
        pair = tuple(sorted((getattr(row, source_col), getattr(row, target_col))))
        anger = float(getattr(row, anger_col))
        history = histories[pair]

#Usuwamy z historii interakcje, które są starsze niż okno czasowe (24h) w stosunku do aktualnego timestampu
        while history and timestamp - history[0][0] > window:
            history.popleft() 

        anger_values = [item[1] for item in history]
        counts.append(len(history))
        mean_anger.append(float(sum(anger_values) / len(anger_values)) if anger_values else 0.0)
        max_anger.append(float(max(anger_values)) if anger_values else 0.0)

        history.append((timestamp, anger))

    result["prev_pair_interactions_24h"] = counts
    result["prev_pair_mean_anger_24h"] = mean_anger
    result["prev_pair_max_anger_24h"] = max_anger
    return result





## Metoda

1. Sortujemy rekordy po czasie.
2. Dla każdej skierowanej pary subredditów liczymy wcześniejsze interakcje z ostatnich 24 godzin.
3. Wysoki anger definiujemy jako średni wcześniejszy `LIWC_Anger` co najmniej na poziomie 75 percentyla wśród rekordów z jakąkolwiek wcześniejszą interakcją.
4. Porównujemy odsetek negatywnych linków i liczymy test Fishera oraz iloraz szans.


## Testowanie hipotezy eskalacji emocjonalnej

Kod wczytuje dane, tworzy cechy opisujące wcześniejsze interakcje pomiędzy tą samą parą subredditów w ostatnich 24 godzinach oraz oznacza, czy aktualny link jest negatywny.

Następnie spośród rekordów posiadających wcześniejszą historię wyznaczany jest próg **wysokiego wcześniejszego poziomu złości**. Za wysoki poziom uznawane są wartości średniego `LIWC_Anger` należące do górnych 25% obserwacji.

Dla każdego aktualnego linku kod sprawdza więc:

* czy pomiędzy tą samą parą subredditów występowały wcześniejsze interakcje,
* czy ich średni poziom złości był wysoki,
* czy aktualny link jest negatywny.

Na końcu tworzona jest tabela porównująca linki po wysokiej wcześniejszej złości z pozostałymi linkami. Test Fishera oraz test chi-kwadrat pozwalają sprawdzić, czy różnica w występowaniu negatywnych linków jest istotna statystycznie.

* `odds_ratio > 1` oznacza, że negatywne linki częściej występują po wysokiej wcześniejszej złości.
* `p-value < 0.05` oznacza, że wynik można uznać za istotny statystycznie.


In [3]:
df = pd.read_csv(DATA_PATH)
df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"])

features = add_pair_history_features(df)
features["is_negative_link"] = sentiment_to_binary_negative(features["LINK_SENTIMENT"])

history_rows = features[features["prev_pair_interactions_24h"] > 0].copy()
anger_threshold = history_rows["prev_pair_mean_anger_24h"].quantile(0.75)

features["high_previous_anger_24h"] = (
    (features["prev_pair_interactions_24h"] > 0)
    & (features["prev_pair_mean_anger_24h"] >= anger_threshold)
)

table = pd.crosstab(features["high_previous_anger_24h"], features["is_negative_link"])
table = table.reindex(index=[False, True], columns=[0, 1], fill_value=0)
display(table)

odds_ratio, fisher_p = stats.fisher_exact(table.to_numpy())
chi2, chi2_p, _, _ = stats.chi2_contingency(table.to_numpy())
print("Próg wysokiego anger:", anger_threshold)
print("Odds ratio:", odds_ratio)
print("Fisher p-value:", fisher_p)
print("Chi2 p-value:", chi2_p)


is_negative_link,0,1
high_previous_anger_24h,,
False,45950,3844
True,114,10


Próg wysokiego anger: 0.0064516129032258
Odds ratio: 1.0485696551472334
Fisher p-value: 0.8656127056346432
Chi2 p-value: 1.0


In [4]:
rates = (
    features.groupby("high_previous_anger_24h")["is_negative_link"]
    .agg(rows="size", negative_links="sum", negative_rate="mean")
    .reset_index()
)
display(rates)

rates.to_csv(OUTPUT_DIR / "notebook_h1_rates.csv", index=False)


,high_previous_anger_24h,rows,negative_links,negative_rate
0,False,49794,3844,0.077198
1,True,124,10,0.080645


## Wniosek

W uruchomionej analizie rdzeniowej wynik nie wspiera H1: grupa z wysokim wcześniejszym `LIWC_Anger` ma bardzo podobny odsetek linków negatywnych do pozostałych rekordów, a test Fishera nie wskazuje istotnej różnicy.

Interpretacyjnie ważne jest też to, że tylko niewielka liczba rekordów ma historię tej samej pary w poprzednich 24 godzinach. Hipotezę warto powtórzyć także dla okien 48h i 7 dni albo dla par nieskierowanych.
